In [2]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [3]:
file_path = "positions.csv"
data = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "nikitricky/chess-positions",
  file_path
)

Using Colab cache for faster access to the 'chess-positions' dataset.


/usr/local/lib/python3.12/dist-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [4]:
#print("First 5 records:", data.head())

First 5 records:                                                  fen playing  score  mate  \
0  rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...    e2e4  -35.0   NaN   
1  rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...    e7e6   36.0   NaN   
2  rnbqkbnr/pppp1ppp/4p3/8/3PP3/8/PPP2PPP/RNBQKBN...    d2d4  -27.0   NaN   
3  rnbqkbnr/p1pp1ppp/1p2p3/8/3PP3/8/PPP2PPP/RNBQK...    b7b6   81.0   NaN   
4  rnbqkbnr/p1pp1ppp/1p2p3/8/3PP3/P7/1PP2PPP/RNBQ...    a2a3  -65.0   NaN   

   depth   game_id        date      time  white    black white_result  \
0     20  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   
1     22  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   
2     24  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   
3     20  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   
4     22  j1dkb5dw  2012.12.31  23:01:03  BFG9k  mamalak            1   

  black_result white_elo black_elo                           opening  \
0        

In [5]:
print("\nDataFrame Info:")
data.info()

print("\nDataFrame Description:")
data.describe()


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1091078 entries, 0 to 1091077
Data columns (total 17 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   fen           1091078 non-null  object 
 1   playing       1091078 non-null  object 
 2   score         981566 non-null   float64
 3   mate          109512 non-null   float64
 4   depth         1091078 non-null  int64  
 5   game_id       1091078 non-null  object 
 6   date          1091078 non-null  object 
 7   time          1091078 non-null  object 
 8   white         1091078 non-null  object 
 9   black         1091078 non-null  object 
 10  white_result  1091078 non-null  object 
 11  black_result  1091078 non-null  object 
 12  white_elo     1091078 non-null  object 
 13  black_elo     1091078 non-null  object 
 14  opening       1091078 non-null  object 
 15  time_control  1091078 non-null  object 
 16  termination   1091078 non-null  object 
dtypes: float64

,score,mate,depth
count,981566.000000,109512.000000,1.091078e+06
mean,20.746932,0.587881,3.327937e+01
std,651.694704,9.072899,4.161055e+01
min,-20000.000000,-71.000000,0.000000e+00
25%,-129.000000,-5.000000,2.200000e+01
50%,7.000000,1.000000,2.300000e+01
75%,197.000000,6.000000,2.500000e+01
max,20000.000000,70.000000,2.450000e+02


In [6]:
data_processed = data.copy()

# Convert 'white_elo' and 'black_elo' to numeric, coercing errors
data_processed['white_elo'] = pd.to_numeric(data_processed['white_elo'], errors='coerce')
data_processed['black_elo'] = pd.to_numeric(data_processed['black_elo'], errors='coerce')

# Identify numerical columns for normalization (including the converted ELOs)
numerical_cols = ['score', 'mate', 'depth', 'white_elo', 'black_elo']

# Fill missing values with the median for each numerical column
for col in numerical_cols:
    if data_processed[col].isnull().any():
        median_val = data_processed[col].median()
        data_processed[col].fillna(median_val, inplace=True)

print("\nMissing values after imputation:")
print(data_processed[numerical_cols].isnull().sum())

print("\nData types after conversion:")
print(data_processed[numerical_cols].dtypes)


Missing values after imputation:
score        0
mate         0
depth        0
white_elo    0
black_elo    0
dtype: int64

Data types after conversion:
score        float64
mate         float64
depth          int64
white_elo    float64
black_elo    float64
dtype: object


/tmp/ipykernel_50357/3908162076.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_processed[col].fillna(median_val, inplace=True)


Now that we have handled the data types and missing values, we can apply a `StandardScaler` to normalize these numerical features. This will transform the data so that it has a mean of 0 and a standard deviation of 1, which is often beneficial for machine learning models.

In [7]:
from sklearn.preprocessing import StandardScaler

# Initialize StandardScaler
scaler = StandardScaler()

# Apply StandardScaler to the numerical columns
data_processed[numerical_cols] = scaler.fit_transform(data_processed[numerical_cols])

print("\nFirst 5 rows of normalized numerical features:")
display(data_processed[numerical_cols].head())

print("\nDescriptive statistics of normalized numerical features:")
display(data_processed[numerical_cols].describe())


First 5 rows of normalized numerical features:


,score,mate,depth,white_elo,black_elo
0,-0.087953,0.014377,-0.319135,0.109925,-0.999519
1,0.026908,0.014377,-0.271070,0.109925,-0.999519
2,-0.075011,0.014377,-0.223005,0.109925,-0.999519
3,0.099707,0.014377,-0.319135,0.109925,-0.999519
4,-0.136486,0.014377,-0.271070,0.109925,-0.999519



Descriptive statistics of normalized numerical features:


,score,mate,depth,white_elo,black_elo
count,1.091078e+06,1.091078e+06,1.091078e+06,1.091078e+06,1.091078e+06
mean,4.288350e-18,-1.048480e-18,1.676266e-17,-2.500723e-16,-1.566078e-16
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-3.238656e+01,-2.501112e+01,-7.997824e-01,-3.738753e+00,-3.897766e+00
25%,-1.753122e-01,1.437736e-02,-2.710701e-01,-6.179340e-01,-6.198150e-01
50%,-2.000709e-02,1.437736e-02,-2.470377e-01,-2.966456e-02,-5.746844e-02
75%,2.113328e-01,1.437736e-02,-1.989729e-01,6.433555e-01,6.827141e-01
max,3.232390e+01,2.399715e+01,5.088150e+00,3.764175e+00,3.657864e+00


In [8]:
# Define features (X) and target (y)
# The target 'white_result' needs to be converted to a numerical type (e.g., 0 or 1).
# Assuming 'white_result' is '1' for white win and '0' for anything else.
# Let's inspect the unique values of 'white_result' first.
print("Unique values in 'white_result' before conversion:", data_processed['white_result'].unique())

# Convert 'white_result' to a numerical format. Assuming '1' means white won, '0' means white lost/draw.
# Based on the head() output in HnZ7gUGqnn7E, 'white_result' contains '1' and '0'.
# If there are other values (e.g., '1/2' for draw), they should be handled.
# Let's assume '1' is win for white, '0' is loss/draw for white.
# We might need to map '1/2' if it exists to 0 or 0.5 depending on interpretation.

# For simplicity, let's map '1' to 1, and everything else to 0 for binary classification.
y = data_processed['white_result'].apply(lambda x: 1 if x == '1' else 0)

X = data_processed[numerical_cols]

print("\nFirst 5 rows of features (X):")
display(X.head())
print("\nFirst 5 rows of target (y):")
display(y.head())

Unique values in 'white_result' before conversion: ['1' '0' '1/2']

First 5 rows of features (X):


,score,mate,depth,white_elo,black_elo
0,-0.087953,0.014377,-0.319135,0.109925,-0.999519
1,0.026908,0.014377,-0.271070,0.109925,-0.999519
2,-0.075011,0.014377,-0.223005,0.109925,-0.999519
3,0.099707,0.014377,-0.319135,0.109925,-0.999519
4,-0.136486,0.014377,-0.271070,0.109925,-0.999519



First 5 rows of target (y):


,white_result
0,1
1,1
2,1
3,1
4,1


Now, we'll split the data into training and testing sets to evaluate the model's performance on unseen data. Then, we will train a `LogisticRegression` model and make predictions.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")

# Initialize and train the Logistic Regression model
# Set max_iter for convergence, especially with a large dataset
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
log_reg_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = log_reg_model.predict(X_test)

# Evaluate the model
print("\nModel Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Training set size: 763754 samples
Testing set size: 327324 samples

Model Accuracy: 0.6416791924820667

Classification Report:
               precision    recall  f1-score   support

           0       0.64      0.67      0.66    167081
           1       0.64      0.61      0.63    160243

    accuracy                           0.64    327324
   macro avg       0.64      0.64      0.64    327324
weighted avg       0.64      0.64      0.64    327324



In [10]:
import pandas as pd

# Get feature names from the X DataFrame
feature_names = X.columns

# Get the coefficients from the trained logistic regression model
coefficients = log_reg_model.coef_[0]

# Create a DataFrame for feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Absolute_Coefficient': abs(coefficients)
})

# Sort by absolute coefficient value to see most important features
feature_importance = feature_importance.sort_values(by='Absolute_Coefficient', ascending=False)

print("\nFeature Importance Table:")
display(feature_importance)


Feature Importance Table:


,Feature,Coefficient,Absolute_Coefficient
4,black_elo,-0.880895,0.880895
3,white_elo,0.832712,0.832712
2,depth,-0.016288,0.016288
0,score,-0.004136,0.004136
1,mate,0.001265,0.001265
